# **Técnicas de suavizamiento exponencial**


In [ ]:
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.api import ExponentialSmoothing, SimpleExpSmoothing, Holt
import matplotlib.pyplot as plt


In [ ]:
url = 'https://raw.githubusercontent.com/JBrianAlicorp/Business-Analytics/master/international-airline-passengers.csv'
airpax_data = pd.read_csv(url,encoding='latin1')
airpax_data.columns = ['Month','passengers']
airpax_data['Month'] = pd.to_datetime(airpax_data['Month'],infer_datetime_format=True) #convert from string to datetime
airpax_data = airpax_data.set_index(['Month'])
airpax_data.head()

In [ ]:
## plot graph
plt.xlabel('Date')
plt.ylabel('Q air passengers')
plt.plot(airpax_data)

## Simple exponencial smoothing (SES)

In [ ]:
HWES1 = SimpleExpSmoothing(airpax_data, initialization_method="estimated").fit()
HWES1_fitted = HWES1.fittedvalues
HWES1_fitted.name = 'Simple Exponencial Smoothing'
pd.concat([airpax_data, HWES1_fitted], axis=1).plot(title='Single Exponential Smoothing')

In [ ]:
results = pd.DataFrame(
    index=["SSE", "AIC", "BIC"])
results["Simple Exponencial"] = [HWES1.sse] + [HWES1.aic] + [HWES1.bic]

In [ ]:
print(HWES1.summary())

## Double exponencial smoothing (DES)

In [ ]:
HWES2_ADD = ExponentialSmoothing(airpax_data,trend='add').fit()#.fittedvalues
HWES2_MUL = ExponentialSmoothing(airpax_data,trend='mul').fit()#.fittedvalues
HWES2_ADD_fitted = HWES2_ADD.fittedvalues
HWES2_MUL_fitted = HWES2_MUL.fittedvalues
HWES2_ADD_fitted.name = 'Additive Double Exp. Smoothing'
HWES2_MUL_fitted.name = 'multiplicative Double Exp. Smoothin'
pd.concat([airpax_data, HWES2_ADD_fitted, HWES2_MUL_fitted], axis=1).plot(title='Double Exponential Smoothing: Additive and Multiplicative Trend');

In [ ]:
results["Double Exp. - Additive"] = [HWES2_ADD.sse] + [HWES2_ADD.aic] + [HWES2_ADD.bic]
results["Double Exp. - Multiplicative"] = [HWES2_MUL.sse] + [HWES2_MUL.aic] + [HWES2_MUL.bic]

In [ ]:
HWES2_ADD.summary()

## Triple exponencial smoothing (TES)

In [ ]:
HWES3_ADD = ExponentialSmoothing(airpax_data,trend='add',seasonal='add',seasonal_periods=12).fit()
HWES3_MUL = ExponentialSmoothing(airpax_data,trend='mul',seasonal='mul',seasonal_periods=12).fit()

HWES3_ADD_fitted = HWES3_ADD.fittedvalues
HWES3_MUL_fitted = HWES3_MUL.fittedvalues
HWES3_ADD_fitted.name = 'Additive TES'
HWES3_MUL_fitted.name = 'multiplicative TES'
pd.concat([airpax_data, HWES3_ADD_fitted, HWES3_MUL_fitted], axis=1).plot(title='Triple Exponential Smoothing: Additive and Multiplicative Trend');

In [ ]:
results["Triple Exp. - Additive"] = [HWES3_ADD.sse] + [HWES3_ADD.aic] + [HWES3_ADD.bic]
results["Triple Exp. - Multiplicative"] = [HWES3_MUL.sse] + [HWES3_MUL.aic] + [HWES3_MUL.bic]

In [ ]:
results

In [ ]:
pd.concat([airpax_data, HWES3_MUL_fitted], axis=1).plot(title='Original and fitted ts');

In [ ]:
HWES3_MUL.summary()

## Holt-Winters

In [ ]:
fit1 = ExponentialSmoothing(
    airpax_data,
    seasonal_periods=12,
    trend="add",
    seasonal="add",
    initialization_method="estimated",
).fit()

fit2 = ExponentialSmoothing(
    airpax_data,
    seasonal_periods=12,
    trend="add",
    seasonal="mul",
    initialization_method="estimated",
).fit()

fit3 = ExponentialSmoothing(
    airpax_data,
    seasonal_periods=12,
    trend="mul",
    seasonal="add",
    initialization_method="estimated",
).fit()

fit4 = ExponentialSmoothing(
    airpax_data,
    seasonal_periods=12,
    trend="mul",
    seasonal="mul",
    initialization_method="estimated",
).fit()

In [ ]:
results2 = pd.DataFrame(
    index=["SSE", "AIC", "BIC"])

In [ ]:
results2["Holt Trend-Add Seas-Add"] = [fit1.sse] + [fit1.aic] + [fit1.bic]
results2["Holt Trend-Add Seas-Mul"] = [fit2.sse] + [fit2.aic] + [fit2.bic]
results2["Holt Trend-Mul Seas-Add"] = [fit3.sse] + [fit3.aic] + [fit3.bic]
results2["Holt Trend-Mul Seas-Mul"] = [fit4.sse] + [fit4.aic] + [fit4.bic]

In [ ]:
results2

Predicción

In [ ]:
ax = airpax_data.plot(
    figsize=(10, 6),
    marker="o",
    color="red",
    alpha = 0.5
)

ax.set_ylabel("Air passengers")
ax.set_xlabel("Year")
fit4.fittedvalues.plot(ax=ax, style="--", color="green")
fit4.forecast(16).rename("Holt-Winters (mul-Trend-mul-seasonal)").plot(
    ax=ax, style="--", marker="o", color="green", legend=True
)

plt.show()

## Evaluación de los modelos:

In [ ]:
#divide into train and validation set
train = airpax_data[:int(0.75*(len(airpax_data)))]
valid = airpax_data[int(0.75*(len(airpax_data))):]

#plotting the data
train['passengers'].plot()
valid['passengers'].plot()
plt.show()

In [ ]:
fit4 = ExponentialSmoothing(
    train,
    seasonal_periods=12,
    #trend="mul",
    seasonal="mul",
    initialization_method="estimated",
).fit()

In [ ]:
fit4.summary()

In [ ]:
# predict for five months in the furure and MS - month start is the frequency
forecast = fit4.forecast(36)
forecast

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error, mean_squared_log_error
import numpy as np
def evaluate_forecast(y,pred):
    results = pd.DataFrame({'r2_score':r2_score(y, pred),
                           }, index=[0])
    results['mean_absolute_error'] = mean_absolute_error(y, pred)
    results['median_absolute_error'] = median_absolute_error(y, pred)
    results['mse'] = mean_squared_error(y, pred)
    results['msle'] = mean_squared_log_error(y, pred)
    results['rmse'] = np.sqrt(results['mse'])
    return results

In [ ]:
evaluate_forecast(train.passengers, fit4.fittedvalues)

In [ ]:
evaluate_forecast(valid.passengers, forecast)

In [ ]:
# grafica para el entrenamiento
ax = train.plot(
    figsize=(10, 6),
    marker="o",
    color="yellow",
)

ax.set_ylabel("Air passengers")
ax.set_xlabel("Year")
fit4.fittedvalues.rename("Holt-Winters (mul-Trend-mul-seasonal)").plot(
    ax=ax, style="--", marker="o", color="green", legend=True
)

plt.show()

In [ ]:
# grafica para la validacion
ax = valid.plot(
    figsize=(10, 6),
    marker="o",
    color="yellow",
)

ax.set_ylabel("Air passengers")
ax.set_xlabel("Year")
forecast.rename("Holt-Winters (mul-Trend-mul-seasonal)").plot(
    ax=ax, style="--", marker="o", color="green", legend=True
)

plt.show()

# Probando con data transformada

In [ ]:
import numpy as np

ylog = np.log(airpax_data)

#divide into train and validation set
train = ylog[:int(0.75*(len(ylog)))]
valid = ylog[int(0.75*(len(ylog))):]

#plotting the data
train['passengers'].plot()
valid['passengers'].plot()
plt.show()

In [ ]:
fit4 = ExponentialSmoothing(
    train,
    seasonal_periods=12,
    trend="mul",
    seasonal="mul",
    initialization_method="estimated",
).fit()

In [ ]:
fit4.summary()

In [ ]:
# predict for five months in the furure and MS - month start is the frequency
forecast = fit4.forecast(36)
forecast

In [ ]:
evaluate_forecast(train.passengers, fit4.fittedvalues)

In [ ]:
evaluate_forecast(valid.passengers, forecast)

In [ ]:
# grafica para el entrenamiento
ax = train.plot(
    figsize=(10, 6),
    marker="o",
    color="yellow",
)

ax.set_ylabel("Air passengers")
ax.set_xlabel("Year")
fit4.fittedvalues.rename("Holt-Winters (mul-Trend-mul-seasonal)").plot(
    ax=ax, style="--", marker="o", color="green", legend=True
)

plt.show()

In [ ]:
# grafica para la validacion
ax = valid.plot(
    figsize=(10, 6),
    marker="o",
    color="yellow",
)

ax.set_ylabel("Air passengers")
ax.set_xlabel("Year")
forecast.rename("Holt-Winters (mul-Trend-mul-seasonal)").plot(
    ax=ax, style="--", marker="o", color="green", legend=True
)

plt.show()

## Probando con transformación box cox

In [ ]:
# import modules
import numpy as np
from scipy import stats

# transform training data & save lambda value
fitted_data, fitted_lambda = stats.boxcox(airpax_data.passengers)

In [ ]:
fitted_lambda

In [ ]:
import numpy as np
import pandas as pd

ylog = pd.DataFrame(fitted_data, columns=['passengers'])

#divide into train and validation set
train = ylog[:int(0.75*(len(ylog)))]
valid = ylog[int(0.75*(len(ylog))):]

#plotting the data
train['passengers'].plot()
valid['passengers'].plot()
plt.show()

In [ ]:
fit4 = ExponentialSmoothing(
    train,
    seasonal_periods=12,
    trend="mul",
    seasonal="mul",
    initialization_method="estimated",
).fit()

In [ ]:
fit4.summary()

In [ ]:
# predict for five months in the furure and MS - month start is the frequency
forecast = fit4.forecast(36)
forecast

In [ ]:
evaluate_forecast(train.passengers, fit4.fittedvalues)

In [ ]:
evaluate_forecast(valid.passengers, forecast)

In [ ]:
# grafica para el entrenamiento
ax = train.plot(
    figsize=(10, 6),
    marker="o",
    color="yellow",
)

ax.set_ylabel("Air passengers")
ax.set_xlabel("Year")
fit4.fittedvalues.rename("Holt-Winters (mul-Trend-mul-seasonal)").plot(
    ax=ax, style="--", marker="o", color="green", legend=True
)

plt.show()

In [ ]:
# grafica para la validacion
ax = valid.plot(
    figsize=(10, 6),
    marker="o",
    color="yellow",
)

ax.set_ylabel("Air passengers")
ax.set_xlabel("Year")
forecast.rename("Holt-Winters (mul-Trend-mul-seasonal)").plot(
    ax=ax, style="--", marker="o", color="green", legend=True
)

plt.show()

# Encontrando el mejor modelo de suavizado: Data eléctrica

In [ ]:
url = 'https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/h2o.csv'
datos = pd.read_csv(url, sep=',')
datos['fecha'] = pd.to_datetime(datos['fecha'], format='%Y-%m-%d')
datos = datos.set_index('fecha') # fecha como nombre de fila
datos = datos.rename(columns={'x': 'y'})
print(datos)


In [ ]:
#divide into train and validation set
train = datos[:int(0.90*(len(datos)))]
valid = datos[int(0.90*(len(datos))):]

#plotting the data
train['y'].plot()
valid['y'].plot()
plt.show()

In [ ]:
valid.shape

In [ ]:
fit5 = ExponentialSmoothing(
    train,
    seasonal_periods=12,
    trend="mul",
    seasonal="mul",
    initialization_method="estimated",
).fit()

In [ ]:
fit5.summary()

In [ ]:
# predict for five months in the furure and MS - month start is the frequency
forecast = fit5.forecast(21)

In [ ]:
evaluate_forecast(train.y, fit5.fittedvalues)

In [ ]:
evaluate_forecast(valid.y, forecast)

In [ ]:
# grafica para el entrenamiento
ax = train.plot(
    figsize=(10, 6),
    marker="o",
    color="yellow",
)

ax.set_ylabel("Electric")
ax.set_xlabel("Year")
fit5.fittedvalues.rename("Holt-Winters (mul-Trend-mul-seasonal)").plot(
    ax=ax, style="--", marker="o", color="green", legend=True
)

plt.show()

In [ ]:
# grafica para la validacion
ax = valid.plot(
    figsize=(10, 6),
    marker="o",
    color="yellow",
)

ax.set_ylabel("Electric")
ax.set_xlabel("Year")
forecast.rename("Holt-Winters (mul-Trend-mul-seasonal)").plot(
    ax=ax, style="--", marker="o", color="green", legend=True
)

plt.show()